In [1]:
# Import standard libraries
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Load Data and Vocabularies

In [2]:
# Load data and utilities
from src.data_loader import load_data, create_dataloaders, create_bert_dataloaders
from src.utils import build_vocab, load_slot_vocab, load_intent_vocab, compute_slot_f1

# Load datasets
train_data = load_data('dataset/train')
val_data = load_data('dataset/valid')
test_data = load_data('dataset/test')

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")
print(f"\nSample data: {train_data[0]}")

Train samples: 13084
Validation samples: 700
Test samples: 700

Sample data: {'intent_label': 'AddToPlaylist', 'words': ['Add', 'Don', 'and', 'Sherri', 'to', 'my', 'Meditate', 'to', 'Sounds', 'of', 'Nature', 'playlist'], 'slot_labels': ['O', 'B-entity_name', 'I-entity_name', 'I-entity_name', 'O', 'B-playlist_owner', 'B-playlist', 'I-playlist', 'I-playlist', 'I-playlist', 'I-playlist', 'O'], 'length': 12}


In [3]:
# Load vocabularies
intent_vocab = load_intent_vocab('dataset/vocab.intent')
slot_vocab = load_slot_vocab('dataset/vocab.slot')

# Build word vocabulary from training data only
train_sentences = [' '.join(item['words']) for item in train_data]
word_vocab = build_vocab(train_sentences, min_freq=1)

print(f"Intent classes: {len(intent_vocab)}")
print(f"Slot labels: {len(slot_vocab)}")
print(f"Vocabulary size: {len(word_vocab)}")
print(f"\nIntent labels: {list(intent_vocab.keys())}")

Intent classes: 7
Slot labels: 73
Vocabulary size: 13456

Intent labels: ['AddToPlaylist', 'BookRestaurant', 'GetWeather', 'PlayMusic', 'RateBook', 'SearchCreativeWork', 'SearchScreeningEvent']


## Common Configuration

In [4]:
# ── Common Hyperparameters ──────────────────────────────────────────
# Shared across all three models so experiments are comparable.

MAX_LEN      = 50        # Maximum token sequence length
BATCH_SIZE   = 32        # Mini-batch size
MAX_EPOCHS   = 20        # Upper bound on training epochs
LEARNING_RATE = 1e-3     # Default learning rate (overridden per model if needed)
CLIP_GRAD    = 5.0       # Max gradient norm for clipping
PATIENCE     = 5         # Early-stopping patience (epochs without improvement)
MIN_DELTA    = 1e-4      # Minimum val-loss decrease to count as improvement
ALPHA        = 1.0       # Weight for intent classification loss
BETA         = 1.0       # Weight for slot filling loss
DROPOUT      = 0.3       # Dropout rate (GRU models)

# Model-specific learning rates
LR_ENCODER_DECODER = 1e-3
LR_TRANSFORMER     = 5e-4
LR_BERT            = 1e-3

print("=== Training Configuration ===")
print(f"  MAX_LEN      : {MAX_LEN}")
print(f"  BATCH_SIZE   : {BATCH_SIZE}")
print(f"  MAX_EPOCHS   : {MAX_EPOCHS}")
print(f"  PATIENCE     : {PATIENCE}")
print(f"  MIN_DELTA    : {MIN_DELTA}")
print(f"  CLIP_GRAD    : {CLIP_GRAD}")
print(f"  ALPHA / BETA : {ALPHA} / {BETA}")
print(f"  DROPOUT      : {DROPOUT}")
print(f"  LR (Enc-Dec) : {LR_ENCODER_DECODER}")
print(f"  LR (Trans.)  : {LR_TRANSFORMER}")
print(f"  LR (BERT)    : {LR_BERT}")

=== Training Configuration ===
  MAX_LEN      : 50
  BATCH_SIZE   : 32
  MAX_EPOCHS   : 20
  PATIENCE     : 5
  MIN_DELTA    : 0.0001
  CLIP_GRAD    : 5.0
  ALPHA / BETA : 1.0 / 1.0
  DROPOUT      : 0.3
  LR (Enc-Dec) : 0.001
  LR (Trans.)  : 0.0005
  LR (BERT)    : 0.001


## 2. Define Training and Evaluation Functions

In [5]:
def train_joint_model(
    model, 
    train_loader, 
    val_loader, 
    epochs=20, 
    lr=1e-3,
    slot_pad_idx=0,
    alpha=1.0,  # weight for intent loss
    beta=1.0,   # weight for slot loss
    clip_grad=5.0,
    patience=3,  # early stopping patience
    min_delta=1e-4  # minimum change to qualify as improvement
):
    """
    Train a joint intent classification and slot filling model.
    
    Args:
        model: The joint NLU model
        train_loader: Training DataLoader
        val_loader: Validation DataLoader
        epochs: Maximum number of training epochs
        lr: Learning rate
        slot_pad_idx: Padding index for slot labels (ignored in loss)
        alpha: Weight for intent classification loss
        beta: Weight for slot filling loss
        clip_grad: Maximum gradient norm for clipping
        patience: Number of epochs to wait before early stopping
        min_delta: Minimum change in validation loss to qualify as improvement
    """
    model = model.to(device)
    
    # Loss functions
    intent_criterion = nn.CrossEntropyLoss()
    slot_criterion = nn.CrossEntropyLoss(ignore_index=slot_pad_idx)
    
    optimizer = Adam(model.parameters(), lr=lr)
    
    history = {'train_loss': [], 'val_loss': [], 'intent_acc': [], 'slot_f1': []}
    
    # Early stopping variables
    best_val_loss = float('inf')
    epochs_no_improve = 0
    early_stopped = False
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_train_loss = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            intent_labels = batch['intent_label'].to(device)
            slot_labels = batch['slot_labels'].to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            intent_logits, slot_logits = model(input_ids, attention_mask)
            
            # Compute losses
            intent_loss = intent_criterion(intent_logits, intent_labels)
            # Reshape slot logits for cross entropy: (batch * seq_len, num_slots)
            slot_logits_flat = slot_logits.view(-1, slot_logits.size(-1))
            slot_labels_flat = slot_labels.view(-1)
            slot_loss = slot_criterion(slot_logits_flat, slot_labels_flat)
            
            # Combined loss
            total_loss = alpha * intent_loss + beta * slot_loss
            
            # Backward pass
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            
            total_train_loss += total_loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        
        # Validation phase
        val_metrics = evaluate_joint_model(model, val_loader, slot_pad_idx, slot_vocab)
        history['val_loss'].append(val_metrics['total_loss'])
        history['intent_acc'].append(val_metrics['intent_accuracy'])
        history['slot_f1'].append(val_metrics['slot_f1'])
        
        print(f"Epoch {epoch+1}/{epochs}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {val_metrics['total_loss']:.4f}")
        print(f"  Intent Accuracy: {val_metrics['intent_accuracy']:.4f}")
        print(f"  Slot F1: {val_metrics['slot_f1']:.4f}")
        print(f"  Joint Accuracy: {val_metrics['joint_accuracy']:.4f}")
        
        # Early stopping check
        if val_metrics['total_loss'] < best_val_loss - min_delta:
            best_val_loss = val_metrics['total_loss']
            epochs_no_improve = 0
            print(f"  ✓ New best validation loss: {best_val_loss:.4f}")
        else:
            epochs_no_improve += 1
            print(f"  No improvement for {epochs_no_improve} epoch(s)")
            
            if epochs_no_improve >= patience:
                print(f"\nEarly stopping triggered after {epoch+1} epochs!")
                early_stopped = True
                break
        
        print()
    
    if not early_stopped:
        print(f"Training completed all {epochs} epochs")
    
    return history


In [6]:
def evaluate_joint_model(model, data_loader, slot_pad_idx=0, slot_vocab=None):
    """
    Evaluate a joint intent classification and slot filling model.
    
    Returns:
        Dictionary with metrics: total_loss, intent_accuracy, slot_f1, joint_accuracy
    """
    model = model.to(device)
    model.eval()
    
    intent_criterion = nn.CrossEntropyLoss()
    slot_criterion = nn.CrossEntropyLoss(ignore_index=slot_pad_idx)
    
    total_loss = 0
    intent_correct = 0
    intent_total = 0
    joint_correct = 0
    
    all_slot_preds = []
    all_slot_labels = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            intent_labels = batch['intent_label'].to(device)
            slot_labels = batch['slot_labels'].to(device)
            
            # Forward pass
            intent_logits, slot_logits = model(input_ids, attention_mask)
            
            # Compute losses
            intent_loss = intent_criterion(intent_logits, intent_labels)
            slot_logits_flat = slot_logits.view(-1, slot_logits.size(-1))
            slot_labels_flat = slot_labels.view(-1)
            slot_loss = slot_criterion(slot_logits_flat, slot_labels_flat)
            total_loss += (intent_loss + slot_loss).item()
            
            # Intent predictions
            intent_preds = torch.argmax(intent_logits, dim=1)
            intent_correct += (intent_preds == intent_labels).sum().item()
            intent_total += intent_labels.size(0)
            
            # Slot predictions
            slot_preds = torch.argmax(slot_logits, dim=2)  # (batch, seq_len)
            
            # Collect for F1 computation
            for i in range(slot_preds.size(0)):
                pred_seq = slot_preds[i].cpu().tolist()
                label_seq = slot_labels[i].cpu().tolist()
                all_slot_preds.append(pred_seq)
                all_slot_labels.append(label_seq)
                
                # Joint accuracy: both intent and all slots correct
                intent_match = (intent_preds[i] == intent_labels[i]).item()
                # Compare only non-padding slots
                slot_match = all(
                    p == l for p, l in zip(pred_seq, label_seq) 
                    if l != slot_pad_idx
                )
                if intent_match and slot_match:
                    joint_correct += 1
    
    # Compute metrics
    avg_loss = total_loss / len(data_loader)
    intent_accuracy = intent_correct / intent_total
    joint_accuracy = joint_correct / intent_total
    
    # Compute slot F1
    if slot_vocab is not None:
        _, _, slot_f1 = compute_slot_f1(
            all_slot_preds, 
            all_slot_labels, 
            slot_vocab, 
            ignore_index=slot_pad_idx
        )
    else:
        slot_f1 = 0.0
    
    return {
        'total_loss': avg_loss,
        'intent_accuracy': intent_accuracy,
        'slot_f1': slot_f1,
        'joint_accuracy': joint_accuracy
    }

## 3. Model 1: Encoder-Decoder with Learned Embeddings

In [7]:
from src.models import EncoderDecoderNLU

# Create DataLoaders for Model 1 & 2 (uses config values)
train_loader, val_loader, test_loader = create_dataloaders(
    train_data, val_data, test_data,
    word_vocab, slot_vocab, intent_vocab,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Using device: cuda
Training batches: 409
Validation batches: 22
Test batches: 22


In [8]:
# Initialize Model 1
model1 = EncoderDecoderNLU(
    vocab_size=len(word_vocab),
    embedding_dim=300,
    hidden_dim=128,
    num_intents=len(intent_vocab),
    num_slots=len(slot_vocab),
    n_layers=1,
    dropout=DROPOUT,
    pad_idx=word_vocab['<PAD>']
)

print(f"Model 1 parameters: {sum(p.numel() for p in model1.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model1.parameters() if p.requires_grad):,}")

Model 1 parameters: 4,608,272
Trainable parameters: 4,608,272


In [9]:
# Train Model 1 (uses global config)
history1 = train_joint_model(
    model1,
    train_loader,
    val_loader,
    epochs=MAX_EPOCHS,
    lr=LR_ENCODER_DECODER,
    slot_pad_idx=slot_vocab['<PAD>'],
    alpha=ALPHA,
    beta=BETA,
    clip_grad=CLIP_GRAD,
    patience=PATIENCE,
    min_delta=MIN_DELTA
)

Evaluating: 100%|██████████| 22/22 [00:00<00:00, 85.68it/s]


Epoch 1/20:
  Train Loss: 1.5260
  Val Loss: 0.5152
  Intent Accuracy: 0.9771
  Slot F1: 0.7956
  Joint Accuracy: 0.4614
  ✓ New best validation loss: 0.5152



Evaluating: 100%|██████████| 22/22 [00:00<00:00, 58.50it/s]


Epoch 2/20:
  Train Loss: 0.4774
  Val Loss: 0.3384
  Intent Accuracy: 0.9857
  Slot F1: 0.8637
  Joint Accuracy: 0.6171
  ✓ New best validation loss: 0.3384



Evaluating: 100%|██████████| 22/22 [00:00<00:00, 146.83it/s]


...

Epoch 11/20:
  Train Loss: 0.0855
  Val Loss: 0.3867
  Intent Accuracy: 0.9829
  Slot F1: 0.9031
  Joint Accuracy: 0.6986
  No improvement for 3 epoch(s)



Evaluating: 100%|██████████| 22/22 [00:00<00:00, 120.77it/s]


Epoch 12/20:
  Train Loss: 0.0780
  Val Loss: 0.3542
  Intent Accuracy: 0.9829
  Slot F1: 0.9018
  Joint Accuracy: 0.6871
  No improvement for 4 epoch(s)



Evaluating: 100%|██████████| 22/22 [00:00<00:00, 122.53it/s]

Epoch 13/20:
  Train Loss: 0.0712
  Val Loss: 0.3753
  Intent Accuracy: 0.9871
  Slot F1: 0.8946
  Joint Accuracy: 0.6571
  No improvement for 5 epoch(s)

Early stopping triggered after 13 epochs!


In [10]:
# Evaluate Model 1 on test set
print("Model 1 - Test Set Evaluation:")
test_metrics1 = evaluate_joint_model(model1, test_loader, slot_vocab['<PAD>'], slot_vocab)
print(f"  Intent Accuracy: {test_metrics1['intent_accuracy']:.4f}")
print(f"  Slot F1: {test_metrics1['slot_f1']:.4f}")
print(f"  Joint Accuracy: {test_metrics1['joint_accuracy']:.4f}")

Model 1 - Test Set Evaluation:


Evaluating: 100%|██████████| 22/22 [00:00<00:00, 98.10it/s] 

  Intent Accuracy: 0.9757
  Slot F1: 0.8791
  Joint Accuracy: 0.6200


## 4. Model 2: Transformer with Multi-Head Attention

In [11]:
from src.models import TransformerNLU

# Initialize Model 2
model2 = TransformerNLU(
    vocab_size=len(word_vocab),
    d_model=256,
    nhead=8,
    num_encoder_layers=4,
    dim_feedforward=1024,
    num_intents=len(intent_vocab),
    num_slots=len(slot_vocab),
    max_len=MAX_LEN,
    dropout=0.1,
    pad_idx=word_vocab['<PAD>']
)

print(f"Model 2 parameters: {sum(p.numel() for p in model2.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model2.parameters() if p.requires_grad):,}")

Model 2 parameters: 6,656,336
Trainable parameters: 6,656,336


In [12]:
# Train Model 2 (uses global config)
history2 = train_joint_model(
    model2,
    train_loader,
    val_loader,
    epochs=MAX_EPOCHS,
    lr=LR_TRANSFORMER,
    slot_pad_idx=slot_vocab['<PAD>'],
    alpha=ALPHA,
    beta=BETA,
    clip_grad=CLIP_GRAD,
    patience=PATIENCE,
    min_delta=MIN_DELTA
)

Epoch 1/20 [Train]:   0%|          | 0/409 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]/home/bharath/Documents/nlp/project/ICSF/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
Evaluating: 100%|██████████| 22/22 [00:00<00:00, 78.75it/s]


Epoch 1/20:
  Train Loss: 1.0948
  Val Loss: 0.5012
  Intent Accuracy: 0.9871
  Slot F1: 0.7739
  Joint Accuracy: 0.3800
  ✓ New best validation loss: 0.5012



Evaluating: 100%|██████████| 22/22 [00:00<00:00, 157.35it/s]


Epoch 2/20:
  Train Loss: 0.3582
  Val Loss: 0.4881
  Intent Accuracy: 0.9857
  Slot F1: 0.8276
  Joint Accuracy: 0.4729
  ✓ New best validation loss: 0.4881



...

Epoch 10/20:
  Train Loss: 0.0630
  Val Loss: 0.4689
  Intent Accuracy: 0.9871
  Slot F1: 0.8659
  Joint Accuracy: 0.6029
  No improvement for 3 epoch(s)



Evaluating: 100%|██████████| 22/22 [00:00<00:00, 93.51it/s]


Epoch 11/20:
  Train Loss: 0.0515
  Val Loss: 0.4973
  Intent Accuracy: 0.9871
  Slot F1: 0.8642
  Joint Accuracy: 0.5814
  No improvement for 4 epoch(s)



Evaluating: 100%|██████████| 22/22 [00:00<00:00, 150.76it/s]

Epoch 12/20:
  Train Loss: 0.0376
  Val Loss: 0.5791
  Intent Accuracy: 0.9814
  Slot F1: 0.8484
  Joint Accuracy: 0.5257
  No improvement for 5 epoch(s)

Early stopping triggered after 12 epochs!


In [13]:
# Evaluate Model 2 on test set
print("Model 2 - Test Set Evaluation:")
test_metrics2 = evaluate_joint_model(model2, test_loader, slot_vocab['<PAD>'], slot_vocab)
print(f"  Intent Accuracy: {test_metrics2['intent_accuracy']:.4f}")
print(f"  Slot F1: {test_metrics2['slot_f1']:.4f}")
print(f"  Joint Accuracy: {test_metrics2['joint_accuracy']:.4f}")

Model 2 - Test Set Evaluation:


Evaluating: 100%|██████████| 22/22 [00:00<00:00, 67.30it/s]

  Intent Accuracy: 0.9857
  Slot F1: 0.8457
  Joint Accuracy: 0.5500


## 5. Model 3: Encoder-Decoder with Frozen BERT Embeddings

In [14]:
from transformers import BertTokenizer
from src.models import BertEncoderDecoderNLU

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

# Create BERT DataLoaders with subword alignment
bert_train_loader, bert_val_loader, bert_test_loader = create_bert_dataloaders(
    train_data, val_data, test_data,
    tokenizer, slot_vocab, intent_vocab,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN
)

print(f"BERT Training batches: {len(bert_train_loader)}")

BERT Training batches: 409


In [15]:
# Initialize Model 3
model3 = BertEncoderDecoderNLU(
    hidden_dim=128,
    num_intents=len(intent_vocab),
    num_slots=len(slot_vocab),
    n_layers=1,
    dropout=DROPOUT,
    bert_model_name='bert-base-cased'
)

total_params = sum(p.numel() for p in model3.parameters())
trainable_params = sum(p.numel() for p in model3.parameters() if p.requires_grad)
print(f"Model 3 total parameters: {total_params:,}")
print(f"Model 3 trainable parameters: {trainable_params:,}")
print(f"BERT parameters (frozen): {total_params - trainable_params:,}")

Loading BERT model 'bert-base-cased'... This may take a moment.
BERT model loaded and frozen successfully!
Model 3 total parameters: 109,420,880
Model 3 trainable parameters: 1,110,608
BERT parameters (frozen): 108,310,272


In [18]:
# Evaluation function for BERT model (must be defined before train_bert_model)
def evaluate_bert_model(model, data_loader, slot_vocab):
    """Evaluate BERT model with all metrics (matches evaluate_joint_model output)."""
    model = model.to(device)
    model.eval()

    intent_criterion = nn.CrossEntropyLoss()
    slot_criterion = nn.CrossEntropyLoss(ignore_index=-100)

    total_loss = 0
    intent_correct = 0
    intent_total = 0
    joint_correct = 0

    all_slot_preds = []
    all_slot_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            intent_labels = batch['intent_label'].to(device)
            slot_labels = batch['slot_labels'].to(device)

            intent_logits, slot_logits = model(input_ids, attention_mask)

            intent_loss = intent_criterion(intent_logits, intent_labels)
            slot_logits_flat = slot_logits.view(-1, slot_logits.size(-1))
            slot_labels_flat = slot_labels.view(-1)
            slot_loss = slot_criterion(slot_logits_flat, slot_labels_flat)
            total_loss += (intent_loss + slot_loss).item()

            intent_preds = torch.argmax(intent_logits, dim=1)
            intent_correct += (intent_preds == intent_labels).sum().item()
            intent_total += intent_labels.size(0)

            slot_preds = torch.argmax(slot_logits, dim=2)

            for i in range(slot_preds.size(0)):
                pred_seq = slot_preds[i].cpu().tolist()
                label_seq = slot_labels[i].cpu().tolist()
                all_slot_preds.append(pred_seq)
                all_slot_labels.append(label_seq)

                intent_match = (intent_preds[i] == intent_labels[i]).item()
                slot_match = all(
                    p == l for p, l in zip(pred_seq, label_seq)
                    if l != -100
                )
                if intent_match and slot_match:
                    joint_correct += 1

    avg_loss = total_loss / len(data_loader)
    intent_accuracy = intent_correct / intent_total
    joint_accuracy = joint_correct / intent_total

    _, _, slot_f1 = compute_slot_f1(
        all_slot_preds, all_slot_labels, slot_vocab, ignore_index=-100
    )

    return {
        'total_loss': avg_loss,
        'intent_accuracy': intent_accuracy,
        'slot_f1': slot_f1,
        'joint_accuracy': joint_accuracy
    }


# Training function for BERT model with early stopping and history tracking
def train_bert_model(
    model, train_loader, val_loader, slot_vocab,
    epochs=MAX_EPOCHS, lr=LR_BERT, clip_grad=CLIP_GRAD,
    alpha=ALPHA, beta=BETA, patience=PATIENCE, min_delta=MIN_DELTA
):
    """Training function for BERT-based model with -100 ignore index, early stopping, and full history."""
    model = model.to(device)

    intent_criterion = nn.CrossEntropyLoss()
    slot_criterion = nn.CrossEntropyLoss(ignore_index=-100)
    optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'intent_acc': [], 'slot_f1': []}

    best_val_loss = float('inf')
    epochs_no_improve = 0
    early_stopped = False

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            intent_labels = batch['intent_label'].to(device)
            slot_labels = batch['slot_labels'].to(device)

            optimizer.zero_grad()
            intent_logits, slot_logits = model(input_ids, attention_mask)

            intent_loss = intent_criterion(intent_logits, intent_labels)
            slot_logits_flat = slot_logits.view(-1, slot_logits.size(-1))
            slot_labels_flat = slot_labels.view(-1)
            slot_loss = slot_criterion(slot_logits_flat, slot_labels_flat)

            total_loss = alpha * intent_loss + beta * slot_loss
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            total_train_loss += total_loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        val_metrics = evaluate_bert_model(model, val_loader, slot_vocab)
        history['val_loss'].append(val_metrics['total_loss'])
        history['intent_acc'].append(val_metrics['intent_accuracy'])
        history['slot_f1'].append(val_metrics['slot_f1'])

        print(f"Epoch {epoch+1}/{epochs}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {val_metrics['total_loss']:.4f}")
        print(f"  Intent Acc: {val_metrics['intent_accuracy']:.4f}")
        print(f"  Slot F1:    {val_metrics['slot_f1']:.4f}")
        print(f"  Joint Acc:  {val_metrics['joint_accuracy']:.4f}")

        if val_metrics['total_loss'] < best_val_loss - min_delta:
            best_val_loss = val_metrics['total_loss']
            epochs_no_improve = 0
            print(f"  ✓ New best validation loss: {best_val_loss:.4f}")
        else:
            epochs_no_improve += 1
            print(f"  No improvement for {epochs_no_improve} epoch(s)")
            if epochs_no_improve >= patience:
                print(f"\nEarly stopping triggered after {epoch+1} epochs!")
                early_stopped = True
                break
        print()

    if not early_stopped:
        print(f"Training completed all {epochs} epochs")

    return history

In [19]:
# Train Model 3 (uses global config + early stopping)
history3 = train_bert_model(
    model3, bert_train_loader, bert_val_loader, slot_vocab,
    epochs=MAX_EPOCHS, lr=LR_BERT, clip_grad=CLIP_GRAD,
    alpha=ALPHA, beta=BETA, patience=PATIENCE, min_delta=MIN_DELTA
)

Evaluating: 100%|██████████| 22/22 [00:03<00:00,  6.65it/s]


Epoch 1/20:
  Train Loss: 0.3389
  Val Loss:   0.2531
  Intent Acc: 0.9757
  Slot F1:    0.9179
  Joint Acc:  0.7229
  ✓ New best validation loss: 0.2531



Evaluating: 100%|██████████| 22/22 [00:03<00:00,  6.51it/s]


Epoch 2/20:
  Train Loss: 0.2498
  Val Loss:   0.2191
  Intent Acc: 0.9757
  Slot F1:    0.9312
  Joint Acc:  0.7600
  ✓ New best validation loss: 0.2191



Evaluating: 100%|██████████| 22/22 [00:03<00:00,  6.36it/s]


...

Epoch 10/20:
  Train Loss: 0.1102
  Val Loss:   0.1804
  Intent Acc: 0.9800
  Slot F1:    0.9487
  Joint Acc:  0.8371
  No improvement for 3 epoch(s)



Evaluating: 100%|██████████| 22/22 [00:02<00:00, 10.90it/s]


Epoch 11/20:
  Train Loss: 0.1060
  Val Loss:   0.1919
  Intent Acc: 0.9829
  Slot F1:    0.9472
  Joint Acc:  0.8286
  No improvement for 4 epoch(s)



Evaluating: 100%|██████████| 22/22 [00:01<00:00, 11.14it/s]

Epoch 12/20:
  Train Loss: 0.0955
  Val Loss:   0.1886
  Intent Acc: 0.9829
  Slot F1:    0.9483
  Joint Acc:  0.8386
  No improvement for 5 epoch(s)

Early stopping triggered after 12 epochs!


In [20]:
# Evaluate Model 3 on test set
print("Model 3 - Test Set Evaluation:")
test_metrics3 = evaluate_bert_model(model3, bert_test_loader, slot_vocab)
print(f"  Intent Accuracy: {test_metrics3['intent_accuracy']:.4f}")
print(f"  Slot F1: {test_metrics3['slot_f1']:.4f}")
print(f"  Joint Accuracy: {test_metrics3['joint_accuracy']:.4f}")

Model 3 - Test Set Evaluation:


Evaluating: 100%|██████████| 22/22 [00:02<00:00, 10.50it/s]

  Intent Accuracy: 0.9843
  Slot F1: 0.9467
  Joint Accuracy: 0.8357


## 6. Model Comparison Summary

In [21]:
import pandas as pd
import json

# Create comparison table
comparison = pd.DataFrame({
    'Model': ['Encoder-Decoder (Learned)', 'Transformer', 'BERT + Encoder-Decoder'],
    'Total Params': [
        f"{sum(p.numel() for p in model1.parameters()):,}",
        f"{sum(p.numel() for p in model2.parameters()):,}",
        f"{sum(p.numel() for p in model3.parameters()):,}"
    ],
    'Trainable Params': [
        f"{sum(p.numel() for p in model1.parameters() if p.requires_grad):,}",
        f"{sum(p.numel() for p in model2.parameters() if p.requires_grad):,}",
        f"{sum(p.numel() for p in model3.parameters() if p.requires_grad):,}"
    ],
    'Intent Acc': [
        f"{test_metrics1['intent_accuracy']:.4f}",
        f"{test_metrics2['intent_accuracy']:.4f}",
        f"{test_metrics3['intent_accuracy']:.4f}"
    ],
    'Slot F1': [
        f"{test_metrics1['slot_f1']:.4f}",
        f"{test_metrics2['slot_f1']:.4f}",
        f"{test_metrics3['slot_f1']:.4f}"
    ],
    'Joint Acc': [
        f"{test_metrics1['joint_accuracy']:.4f}",
        f"{test_metrics2['joint_accuracy']:.4f}",
        f"{test_metrics3['joint_accuracy']:.4f}"
    ],
    'Epochs Trained': [
        len(history1['train_loss']),
        len(history2['train_loss']),
        len(history3['train_loss'])
    ]
})

print("=" * 85)
print("MODEL COMPARISON SUMMARY")
print("=" * 85)
print(comparison.to_string(index=False))

# Save all histories and test metrics for the report notebook
report_data = {
    'history1': history1,
    'history2': history2,
    'history3': history3,
    'test_metrics1': test_metrics1,
    'test_metrics2': test_metrics2,
    'test_metrics3': test_metrics3,
    'config': {
        'MAX_LEN': MAX_LEN, 'BATCH_SIZE': BATCH_SIZE, 'MAX_EPOCHS': MAX_EPOCHS,
        'LR_ENCODER_DECODER': LR_ENCODER_DECODER, 'LR_TRANSFORMER': LR_TRANSFORMER,
        'LR_BERT': LR_BERT, 'CLIP_GRAD': CLIP_GRAD, 'PATIENCE': PATIENCE,
        'MIN_DELTA': MIN_DELTA, 'ALPHA': ALPHA, 'BETA': BETA, 'DROPOUT': DROPOUT
    },
    'model_params': {
        'model1_total': sum(p.numel() for p in model1.parameters()),
        'model1_trainable': sum(p.numel() for p in model1.parameters() if p.requires_grad),
        'model2_total': sum(p.numel() for p in model2.parameters()),
        'model2_trainable': sum(p.numel() for p in model2.parameters() if p.requires_grad),
        'model3_total': sum(p.numel() for p in model3.parameters()),
        'model3_trainable': sum(p.numel() for p in model3.parameters() if p.requires_grad),
    }
}

with open('report_data.json', 'w') as f:
    json.dump(report_data, f, indent=2)
print("\nTraining data saved to report_data.json for report generation.")

MODEL COMPARISON SUMMARY
                    Model Total Params Trainable Params Intent Acc Slot F1 Joint Acc  Epochs Trained
Encoder-Decoder (Learned)    4,608,272        4,608,272     0.9757  0.8791    0.6200              13
              Transformer    6,656,336        6,656,336     0.9857  0.8457    0.5500              12
   BERT + Encoder-Decoder  109,420,880        1,110,608     0.9843  0.9467    0.8357              12

Training data saved to report_data.json for report generation.


## 7. Save Models

In [22]:
# Save model weights
torch.save(model1.state_dict(), 'encoder_decoder_nlu.pth')
torch.save(model2.state_dict(), 'transformer_nlu.pth')

# For Model 3, only save GRU weights (BERT is frozen)
model3_state = {
    'encoder': model3.encoder.state_dict(),
    'decoder': model3.decoder.state_dict(),
    'encoder_to_decoder': model3.encoder_to_decoder.state_dict(),
    'intent_classifier': model3.intent_classifier.state_dict(),
    'slot_classifier': model3.slot_classifier.state_dict(),
}
torch.save(model3_state, 'bert_encoder_decoder_nlu.pth')

print("Models saved successfully!")

Models saved successfully!
